In [ ]:
import pandas as pd
import numpy as np
import time
import utils

from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.metrics import (
    accuracy_score,
    mean_absolute_error,
    mean_squared_error,
    root_mean_squared_error,
    r2_score,
    roc_auc_score,
    precision_recall_curve,
    auc, average_precision_score
)
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.inspection import DecisionBoundaryDisplay

from sklearn.datasets import fetch_openml
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder

# Baseline Imports
from xgboost import XGBClassifier, XGBRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from catboost import CatBoostClassifier, CatBoostRegressor

import torch

from tabpfn import TabPFNClassifier, TabPFNRegressor
#from tabpfn_extensions.post_hoc_ensembles.sklearn_interface import AutoTabPFNClassifier, AutoTabPFNRegressor




In [ ]:
files = [r"data/PI_DataSet.txt", r"data/INI_DataSet.txt", r"data/NRTI_DataSet.txt", r"data/NNRTI_DataSet.txt"]

for file in files:
    print(file)
    print(file.split("/")[-1].strip(".txt") + "_results.csv")

In [ ]:
test =  pd.read_csv("data/INI_DataSet.txt", sep='\t')

drugs = ["CAB", "RAL", "EVG", "DTG", "BIC"]

for drug in drugs:
    print(drug + ": " , test[drug].nunique())

In [ ]:
input_file = files[0]
#thresholds defined by the database for the classes of "susceptible", "partly susceptible", and "resistant"
thresholds = [
    [3, 15],  # FPV
    [3, 15],  # ATV
    [3, 15],  # IDV
    [9, 55],  # LPV
    [3, 6],  # NFV
    [3, 15],  # SQV
    [2, 8],  # TPV
    [10, 90],  # DRV
    [5, 25],  # X3TC
    [2, 6],  # ABC
    [3, 15],  # AZT
    [1.5, 3],  # D4T
    [1.5, 3],  # DDI
    [1.5, 3],  # TDF
    [3, 10],  # EFV
    [3, 10],  # NVP
    [3, 10],  # ETR
    [3, 10],  # RPV
    [2.5, 10],  # BIC
    [4, 13],  # DTG
    [2.5, 10],  # EVG - upper threshold guessed
    [1.5, 10]  # RAL - upper threshold guessed
]

# Define row and column names
index = ["FPV", "ATV", "IDV", "LPV", "NFV", "SQV", "TPV", "DRV",
         "3TC", "ABC", "AZT", "D4T", "DDI", "TDF",
         "EFV", "NVP", "ETR", "RPV", "BIC", "DTG", "EVG", "RAL"]
columns = ["lower", "upper"]

# Create DataFrame
cutoff_df = pd.DataFrame(thresholds, index=index, columns=columns)

# Reading in and processing high quality File

df = pd.read_csv(input_file, sep='\t')
#print(df)
df = df.iloc[:,1:-1]
print(df)

#Checking how much data is available for each drug
#print(df.loc[:,"FPV":"DRV"].count())

#list of current drugs of the dataset
drugs = [drug for drug in list(df.columns) if not drug.startswith("P") ]

#creating the one hot encoding for the features
enc = OneHotEncoder(handle_unknown='error')

enc.fit(df.loc[:,[drug for drug in list(df.columns) if drug.startswith("P")]])


#going through the drugs and splitting them to test and training depending on the drug

results = pd.DataFrame(columns=["Drug",
                                "RMSE",
                                "AUC ROC MC",
                                "AUC PRC MC",
                                "AUC ROC BI",
                                "AUC PRC BI",
                                "Time"])

print(input_file)

for drug in drugs:
    #print(drug)
    tmp_drugs = drugs.copy()
    #print(tmp_drugs)
    tmp_drugs.remove(drug)
    #print(tmp_drugs)
    last_col = list(df.columns)[-1]
    dataframe = df.drop(tmp_drugs, axis=1)

    #print(dataframe.head())

    dataframe = dataframe.dropna()


    '''    # encoding the levels of susceptibility as 0 for susceptible, 1 as resistant
    dataframe.loc[dataframe[drug] < cutoff_df.loc[drug, "lower"], drug + "_level_binary"] = 0
    dataframe.loc[dataframe[drug] >= cutoff_df.loc[drug, "lower"], drug + "_level_binary"] = 1
    '''

    # encoding the levels of susceptibility as 0 for susceptible, 1 as partly resistant and 2 as completely resistant
    dataframe.loc[dataframe[drug] < cutoff_df.loc[drug, "lower"], drug + "_level"] = 0
    dataframe.loc[dataframe[drug] >= cutoff_df.loc[drug, "upper"], drug + "_level"] = 2
    dataframe.loc[(dataframe[drug] >= cutoff_df.loc[drug, "lower"]) & (
                dataframe[drug] < cutoff_df.loc[drug, "upper"]), drug + "_level"] = 1

    #print(dataframe.head())

    X, y = dataframe.drop([drug, drug + "_level"], axis=1), np.array(dataframe[drug + "_level"])

    print(y)

    print(X.shape)
    if X.shape[0] == 0:
        continue

    X_trafo = enc.transform(X).toarray()

    #print(X)


In [ ]:
# Reading in and processing high quality File
df = pd.read_csv(input_file, sep='\t')
#print(df)
df = df.iloc[:,1:-1]
#print(df2)

#Checking how much data is available for each drug
#print(df.loc[:,"FPV":"DRV"].count())

#list of current drugs of the dataset
drugs = [drug for drug in list(df.columns) if not drug.startswith("P") ]

#creating the one hot encoding for the features
enc = OneHotEncoder(handle_unknown='error')

enc.fit(df.loc[:,[drug for drug in list(df.columns) if drug.startswith("P")]])


#going through the drugs and splitting them to test and training depending on the drug

results = pd.DataFrame(columns=["Drug",
                            "Samples",
                            "AUC ROC",
                            "Time",
                            "AUC RF",
                            "AUC XGB",
                            "AUC CatB"])



for drug in drugs:
    #print(drug)
    tmp_drugs = drugs.copy()
    #print(tmp_drugs)
    tmp_drugs.remove(drug)
    #print(tmp_drugs)
    last_col = list(df.columns)[-1]
    dataframe = df.drop(tmp_drugs, axis=1)

    #print(dataframe.head())

    dataframe = dataframe.dropna()

    if drug not in utils.THRESHOLD_INDICES:
        results = pd.concat([pd.DataFrame([[drug, dataframe.shape[0], None, None, None,
                                            None, None]], columns=results.columns),
                             results], ignore_index=True)
        continue

    # encoding the levels of susceptibility as 0 for susceptible, 1 as partly resistant and 2 as completly resistant

    y = utils.get_classes(dataframe, drug, mode="multiclass")

    #print(dataframe.head())

    X = dataframe.drop([drug], axis=1)

    print(X)

    X_trafo = enc.transform(X).toarray()

    print(X_trafo.shape)

    print(y)
    X_train, X_test, y_train, y_test = train_test_split(X_trafo, y, test_size=0.33, random_state=42)


In [ ]:
"""This script calculates the ROC AUC for the prediction of TabPFN, Random Forest, XGBoost, and
CatBoost and saves time in a file for all drugs in the stanford database file"""

# Setup Imports
import pandas as pd
import numpy as np
import time

import sys
import os

#sys.path.append(os.path.abspath('..'))

import utils

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

from scipy.stats import pearsonr

from sklearn.preprocessing import OneHotEncoder

# Baseline Imports

from tabpfn import TabPFNClassifier


# table for the encoding of the resistance testing into three classes: "susceptible", "intermediate-level resistant", "high-level resistant" with lower and upper thresholds

def running_models(input_file, output_file):


    # Reading in and processing high quality File
    df = pd.read_csv(input_file, sep='\t')

    #removing index and summary column
    df = df.iloc[:,1:-1]


    #list of current drugs of the dataset
    drugs = [drug for drug in list(df.columns) if not drug.startswith("P") ]

    #creating the one hot encoding for the features
    enc = OneHotEncoder(handle_unknown='error')

    enc.fit(df.loc[:,[drug for drug in list(df.columns) if drug.startswith("P")]])

    #result dataframe
    results = pd.DataFrame(columns=["Drug",
                                    "Samples",
                                    "Accuracy",
                                    "Pearson",
                                    "F1"
                                    "AUC PRC",
                                    "AUC ROC",
                                    "Time"])



    for drug in drugs:

        tmp_drugs = drugs.copy().remove(drug)

        #getting labels of only needed drug
        #dataframe = df.drop(tmp_drugs, axis=1)

        #print(dataframe)

        dataframe = df.dropna(subset=[drug])

        #If no thresholds for drug available no prediction possible
        if drug not in utils.THRESHOLD_INDICES:
            results = pd.concat([pd.DataFrame([[drug, dataframe.shape[0], None, None, None,
                                                None, None]], columns=results.columns),
                                 results], ignore_index=True)
            continue


        # encoding the levels of susceptibility as 0 for susceptible, 1 as partly resistant and 2 as completly resistant
        y = utils.get_classes(dataframe, drug, mode="multiclass")


        X = dataframe.drop([drug], axis=1)



        #X_trafo = enc.transform(X).toarray()



    #results.to_csv(output_file)

def main():


    files = [r"data/PI_DataSet.txt", r"data/INI_DataSet.txt", r"data/NRTI_DataSet.txt", r"data/NNRTI_DataSet.txt"]

    for file in files:
        running_models(file, "output/" + (file.split("/")[-1].strip(".txt") + "multilabel_results.csv"))

if __name__ == '__main__':
    main()

In [ ]:
files = [r"data/PI_DataSet.txt", r"data/INI_DataSet.txt", r"data/NRTI_DataSet.txt", r"data/NNRTI_DataSet.txt"]

drug = "testdrug"

for file in files:
        print("../prediction_results/" + (file.split("/")[1].split("_")[0] + "_results/" + drug + "_results/" + "Multilabel_prediction") + ".csv")


In [ ]:
drugs = ['FPV', 'ATV', 'IDV', 'LPV', 'NFV', 'SQV', 'TPV', 'DRV']
drug = "FPV"

tmp_drugs = drugs.copy()
tmp_drugs.remove(drug)

print(tmp_drugs)
print(drugs)

In [ ]:
import numpy as np

import utils

drugs = ['FPV', 'ATV', 'IDV', 'LPV', 'NFV', 'SQV', 'TPV', 'DRV']
drug = "FPV"

files = [r"data/PI_DataSet.txt", r"data/INI_DataSet.txt", r"data/NRTI_DataSet.txt", r"data/NNRTI_DataSet.txt"]

input_file = files[0]

print(input_file.split("/")[1].split("_")[0] + "_results/" + drug + "_results/" + "AutoTabPFN_results")

#utils.save_results( np.array([0]), np.array([0]), label= (input_file.split("/")[1].split("_")[0] + "_results/" + drug + "_results/" + "AutoTabPFN_results"))

In [ ]:
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
import numpy as np

#x = np.array([0.9, 1.1, 1.15, 1.0, 1.1, 1.8, 1.9, 1.95, 2.8, 3.4, 3.5, 4.7, 6.1])
#y= np.array([0.125, 0.145, 0.18, 0.23, 0.24, 0.1, 0.11, 0.2, 0.16, 0.12, 0.13, 0.095, 0])

x = np.array([0.9, 1.1, 1.15, 1.0, 1.1, 1.8, 1.9, 1.95, 2.8, 3.4, 3.5, 4.7])
y= np.array([0.125, 0.145, 0.18, 0.23, 0.24, 0.1, 0.11, 0.2, 0.16, 0.12, 0.13, 0.095])

X = x.reshape(-1, 1)

# Fit linear regression
model = LinearRegression()
model.fit(X, y)

# Get slope and intercept
slope = model.coef_[0]
intercept = model.intercept_

print(f"y = {slope:.4f}x + {intercept:.4f}")

# Predict for plotting
x_range = np.linspace(min(x), max(x), 100).reshape(-1, 1)
y_pred = model.predict(x_range)

# Plot
plt.scatter(x, y, color='red', label='Data points')
plt.plot(x_range, y_pred, color='blue', label='Fit: y={:.3f}x+{:.3f}'.format(slope, intercept))
plt.xlabel('X')
plt.ylabel('Y')
plt.legend()
plt.show()

print(r2_score(y, model.predict(X)))


In [ ]:
import pandas as pd
from scipy.io import arff

# code
arff_file = arff.loadarff('./data/Other_datasets/yeast.arff')


df = pd.DataFrame(arff_file[0])


df.head()

In [ ]:
import utils
import numpy as np
import pandas as pd

input_file = r"./data/PI_DataSet.txt"

# Reading in and processing high quality File
df = pd.read_csv(input_file, sep='\t')

#removing index and summary column
df = df.iloc[:,1:-1]


#list of current drugs of the dataset
drugs = [drug for drug in list(df.columns) if not drug.startswith("P") ]

drug = drugs[0]


#if type(drug) != list:
#   drug = [drug]

print(type(drugs))
print(drug)

print(df)

y = utils.get_classes(df, drugs, mode="multiclass")



print(y)

X = df.drop(drugs, axis=1)


print(X)

In [17]:
from sklearn.datasets import make_classification
from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils import shuffle
import numpy as np

from tabpfn import TabPFNClassifier

X, y1 = make_classification(n_samples=10, n_features=100,
                            n_informative=30, n_classes=2,
                            random_state=1)
y2 = shuffle(y1, random_state=1)
y3 = shuffle(y1, random_state=2)
Y = np.vstack((y1, y2, y3)).T
n_samples, n_features = X.shape # 10,100
n_outputs = Y.shape[1] # 3
n_classes = 2

#clf = TabPFNClassifier()

print()

#forest = RandomForestClassifier(random_state=1)
#multi_target_forest = MultiOutputClassifier(clf, n_jobs=2)
#y_pred = multi_target_forest.fit(X, Y).predict(X)

In [9]:
print(y_pred)

[[2 2 0]
 [1 2 1]
 [2 1 0]
 [0 0 2]
 [0 2 1]
 [0 0 2]
 [1 1 0]
 [1 1 1]
 [0 0 2]
 [2 0 0]]


In [28]:
#print(y1)

y_true= y1

#y_pred = Y


if y_true.shape[0] != y_pred.shape[0]:
    raise Exception("True labels do not match predicted labels")

#splt = label.split('/')[:-1]
#sub_filepath = '/'.join(splt)

#print(sub_filepath)

#Path(path + sub_filepath).mkdir(parents=True, exist_ok=True)

data = {"True": y_true}

for i, column in enumerate(y_pred.T):
    data.update( {str(i): column })

#df = pd.DataFrame(data)

df2 = pd.DataFrame(Y)

#f = df2.add_prefix("Pred_")

y_pred = pd.DataFrame(y_pred, columns=["0", "1", "2"])
y_true = pd.DataFrame(Y, columns=["0", "1", "2"])

y_pred_new = y_pred.add_prefix("Pred_")
y_true_new = y_true.add_prefix("True_")

#y_true_new["True_0"] = 4.0

df = pd.concat([y_true_new, y_pred_new], sort=False, axis = 1)

#print(f)
print(y_pred)
print(y_true)

print(df)
#print(df)

   0  1  2
0  2  2  0
1  1  2  1
2  2  1  0
3  0  0  2
4  0  2  1
5  0  0  2
6  1  1  0
7  1  1  1
8  0  0  2
9  2  0  0
   0  1  2
0  2  2  0
1  1  2  1
2  2  1  0
3  0  0  2
4  0  2  1
5  0  0  2
6  1  1  0
7  1  1  1
8  0  0  2
9  2  0  0
   True_0  True_1  True_2  Pred_0  Pred_1  Pred_2
0       2       2       0       2       2       0
1       1       2       1       1       2       1
2       2       1       0       2       1       0
3       0       0       2       0       0       2
4       0       2       1       0       2       1
5       0       0       2       0       0       2
6       1       1       0       1       1       0
7       1       1       1       1       1       1
8       0       0       2       0       0       2
9       2       0       0       2       0       0


In [14]:
import pandas as pd

files = [r"./data/PI_DataSet.txt", r"./data/INI_DataSet.txt", r"./data/NRTI_DataSet.txt", r"./data/NNRTI_DataSet.txt"]

for file in files:
    #running_models(file, "../output/" + (file.split("/")[-1].strip(".txt") + "_multilabel_results.csv"))

    # Reading in and processing high quality File
    df = pd.read_csv(file, sep='\t')

    # removing index and summary column
    df = df.iloc[:, 1:-1]

    # list of current drugs of the dataset
    drugs = [drug for drug in list(df.columns) if not drug.startswith("P")]



    unusable_drugs = [drug for drug in drugs if df[drug].count() <= 10]

    if len(unusable_drugs) > 0:
        df.drop(columns=unusable_drugs, inplace=True)

        drugs = [drug for drug in drugs if drug not in unusable_drugs]

    print(file.split("/")[-1].split("_")[0] + "_results/" + file.split("/")[-1].split("_")[0] + "_Binary_Relevance_MOC_prediction")

PI_results/PI_Binary_Relevance_MOC_prediction
INI_results/INI_Binary_Relevance_MOC_prediction
NRTI_results/NRTI_Binary_Relevance_MOC_prediction
NNRTI_results/NNRTI_Binary_Relevance_MOC_prediction


In [23]:
"""Implementation of the Binary relevance Multilable prediction algorithm using TabPFN and the HIV drug resistance dataset
as an example"""

# Setup Imports
import pandas as pd
import numpy as np
import time

import sys
import os


import utils

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputClassifier

from scipy.stats import pearsonr

from sklearn.preprocessing import OneHotEncoder

# Baseline Imports

from tabpfn import TabPFNClassifier


#files = [r"../data/PI_DataSet.txt", r"../data/INI_DataSet.txt", r"../data/NRTI_DataSet.txt", r"../data/NNRTI_DataSet.txt"]

file =r"./data/INI_DataSet.txt"


#running_models(file, "../output/" + (file.split("/")[-1].strip(".txt") + "_multilabel_results.csv"))

# Reading in and processing high quality File
df = pd.read_csv(file, sep='\t')

# removing index and summary column
df = df.iloc[:, 1:-1]

# list of current drugs of the dataset
drugs = [drug for drug in list(df.columns) if not drug.startswith("P")]

#Filtering out drugs with less than 10 labels present
unusable_drugs = [drug for drug in drugs if df[drug].count() <= 10]

if len(unusable_drugs) > 0:
    df.drop(columns=unusable_drugs, inplace=True)

    drugs = [drug for drug in drugs if drug not in unusable_drugs]

df.dropna(subset=drugs, inplace=True)


# creating the one hot encoding for the features
#enc = OneHotEncoder(handle_unknown='error')

#enc.fit(df.loc[:, [drug for drug in list(df.columns) if drug.startswith("P")]])



X = df.drop(drugs, axis=1)




Y = utils.get_classes(df, drugs, mode="binary")

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.33, random_state=42)

clf = TabPFNClassifier()


multi_target_pfn = MultiOutputClassifier(clf, n_jobs=2)
y_pred = multi_target_pfn.fit(X_train, y_train).predict_proba(X_test)



#BR = BinaryRelevanceTabPFN()


#results = BR.predict(X, Y)




y_pred_df = pd.DataFrame(y_pred, columns=drugs)

y_test_df = pd.DataFrame(y_test, columns=drugs)


utils.save_multilabel(y_pred_df, y_test_df, label= "PI_test", path="./")




ValueError: Must pass 2-d input. shape=(4, 51, 2)

In [24]:
print(y_pred)

[array([[0.749395  , 0.25060502],
       [0.70426   , 0.29574   ],
       [0.01972711, 0.9802729 ],
       [0.75343716, 0.24656287],
       [0.7416516 , 0.2583484 ],
       [0.7108908 , 0.28910914],
       [0.767007  , 0.23299299],
       [0.76973426, 0.2302657 ],
       [0.57962584, 0.42037413],
       [0.04291136, 0.95708865],
       [0.00905313, 0.9909469 ],
       [0.7352188 , 0.26478115],
       [0.7815026 , 0.21849741],
       [0.6709869 , 0.32901317],
       [0.06719641, 0.9328036 ],
       [0.7556479 , 0.24435207],
       [0.7155704 , 0.28442964],
       [0.77232474, 0.22767529],
       [0.01162092, 0.98837906],
       [0.71945304, 0.28054693],
       [0.08596747, 0.9140325 ],
       [0.78320456, 0.21679544],
       [0.02011569, 0.9798843 ],
       [0.0270706 , 0.9729294 ],
       [0.01995885, 0.9800412 ],
       [0.08304584, 0.91695416],
       [0.75883174, 0.24116829],
       [0.7887708 , 0.21122926],
       [0.04778877, 0.95221126],
       [0.02837007, 0.9716299 ],
       [0

In [31]:
#print(y_test)

#utils.save_multilabel_proba(y_pred, y_test, label="Proba_test", path="./")

y_pred_probas = y_pred
y_true = y_test

y_pred_dict = {}

drugs = y_true.columns.values.tolist()

for i, probas in enumerate(y_pred_probas):
    y_pred_dict.update({drugs[i]:probas[:,1]})

y_pred = pd.DataFrame(y_pred_dict)


if y_true.shape != y_pred.shape:
    raise Exception("True labels do not match predicted labels")

#splt = label.split('/')[:-1]
#sub_filepath = '/'.join(splt)

#print(sub_filepath)

#Path(path + sub_filepath).mkdir(parents=True, exist_ok=True)

y_pred_new = y_pred.add_prefix("Pred_Proba_")
y_true_new = y_true.add_prefix("True_")

y_true_new.reset_index(inplace=True, drop=True)

df = pd.concat([y_true_new, y_pred_new], axis=1, sort=False)

print(df)


    True_RAL  True_EVG  True_DTG  True_BIC  Pred_Proba_RAL  Pred_Proba_EVG  \
0        0.0       1.0       0.0       0.0        0.250605        0.662205   
1        0.0       0.0       0.0       0.0        0.295740        0.612352   
2        1.0       1.0       1.0       1.0        0.980273        0.969251   
3        0.0       1.0       0.0       0.0        0.246563        0.604435   
4        1.0       1.0       0.0       0.0        0.258348        0.636171   
5        1.0       1.0       0.0       0.0        0.289109        0.418167   
6        0.0       0.0       0.0       0.0        0.232993        0.444049   
7        0.0       0.0       0.0       0.0        0.230266        0.383306   
8        1.0       1.0       0.0       0.0        0.420374        0.424313   
9        1.0       1.0       1.0       1.0        0.957089        0.914746   
10       1.0       1.0       0.0       0.0        0.990947        0.974317   
11       0.0       0.0       0.0       0.0        0.264781      

In [20]:
ml_res_INI = pd.read_csv("PI_test.csv")

print(ml_res_INI)

    Unnamed: 0  True_RAL  True_EVG  True_DTG  True_BIC  Pred_RAL  Pred_EVG  \
0           15       0.0       1.0       0.0       0.0       0.0       1.0   
1           94       0.0       0.0       0.0       0.0       NaN       NaN   
2          152       1.0       1.0       1.0       1.0       NaN       NaN   
3          105       0.0       1.0       0.0       0.0       NaN       NaN   
4          109       1.0       1.0       0.0       0.0       NaN       NaN   
..         ...       ...       ...       ...       ...       ...       ...   
79          46       NaN       NaN       NaN       NaN       0.0       1.0   
80          47       NaN       NaN       NaN       NaN       0.0       0.0   
81          48       NaN       NaN       NaN       NaN       0.0       0.0   
82          49       NaN       NaN       NaN       NaN       0.0       0.0   
83          50       NaN       NaN       NaN       NaN       1.0       1.0   

    Pred_DTG  Pred_BIC  
0        0.0       0.0  
1        NaN 

In [2]:
import pandas as pd

ml_res_INI = pd.read_csv("./prediction_results/INI_results/INI_Binary_Relevance_MOC_prediction.csv")

ml_res_INI

,Unnamed: 0,True_RAL,True_EVG,True_DTG,True_BIC,Pred_RAL,Pred_EVG,Pred_DTG,Pred_BIC
0,0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
1,1,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,2,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
3,3,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
4,4,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
5,5,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
6,6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,8,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
9,9,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


In [12]:
import utils
import pandas as pd

ml_res_INI = pd.read_csv("./prediction_results/INI_results/INI_Binary_Relevance_MOC_prediction.csv")


y_pred = ml_res_INI.filter(regex="Pred_*")
y_true = ml_res_INI.filter(regex="True_*")


print(utils.subset_acc(y_pred, y_true))

0.0


In [13]:
#print(y_pred.compare(y_true))

#0.627450980392


acc = 0

for i in range(y_pred.shape[0]):
    tmp = 1
    for j in range(y_pred.shape[1]):
        if y_pred.iloc[i,j] != y_true.iloc[i,j]:
            tmp = 0
            break
    acc = acc + tmp


"""
for i in range(y_pred.shape[0]):
    if y_pred.iloc[i,:].eq(y_true.iloc[i,:]).all():
        acc = acc + 1
"""
acc = acc/y_pred.shape[0]

print(acc)


0.6274509803921569


In [22]:
from sklearn.metrics import f1_score



print(f1_score(y_true, y_pred, average="weighted", zero_division=0.0))




0.8386510845357398


In [34]:
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.model_selection import cross_val_predict, cross_val_score, cross_validate
from sklearn import linear_model

# Setup Imports
import pandas as pd
import numpy as np
import time

import sys
import os


import utils

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputClassifier

from scipy.stats import pearsonr

from sklearn.preprocessing import OneHotEncoder

# Baseline Imports

from tabpfn import TabPFNClassifier


#files = [r"../data/PI_DataSet.txt", r"../data/INI_DataSet.txt", r"../data/NRTI_DataSet.txt", r"../data/NNRTI_DataSet.txt"]

file =r"./data/INI_DataSet.txt"


#running_models(file, "../output/" + (file.split("/")[-1].strip(".txt") + "_multilabel_results.csv"))

# Reading in and processing high quality File
df = pd.read_csv(file, sep='\t')

# removing index and summary column
df = df.iloc[:, 1:-1]

# list of current drugs of the dataset
drugs = [drug for drug in list(df.columns) if not drug.startswith("P")]

#Filtering out drugs with less than 10 labels present
unusable_drugs = [drug for drug in drugs if df[drug].count() <= 10]

if len(unusable_drugs) > 0:
    df.drop(columns=unusable_drugs, inplace=True)

    drugs = [drug for drug in drugs if drug not in unusable_drugs]

df.dropna(subset=drugs, inplace=True)


# creating the one hot encoding for the features
#enc = OneHotEncoder(handle_unknown='error')

#enc.fit(df.loc[:, [drug for drug in list(df.columns) if drug.startswith("P")]])



#X = df.drop(drugs, axis=1)




#Y = utils.get_classes(df, drugs, mode="binary")


kf = KFold(n_splits=5, random_state=42, shuffle=True)

clf = TabPFNClassifier()

multi_target_pfn = MultiOutputClassifier(clf)

y_pred = cross_val_predict(multi_target_pfn, X, Y, cv=kf, method="predict_proba")

print(y_pred)

"""
fortrain, test in kf.split(X, Y):
    print("%s %s" % (train, test))
"""
#print(X)

#print(Y)



"""
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.33, random_state=42)

clf = TabPFNClassifier()


multi_target_pfn = MultiOutputClassifier(clf, n_jobs=2)
y_pred = multi_target_pfn.fit(X_train, y_train).predict_proba(X_test)



#BR = BinaryRelevanceTabPFN()


#results = BR.predict(X, Y)




y_pred_df = pd.DataFrame(y_pred, columns=drugs)

y_test_df = pd.DataFrame(y_test, columns=drugs)


utils.save_multilabel(y_pred_df, y_test_df, label= "PI_test", path="./")

"""




[array([[0.28089178, 0.7191082 ],
       [0.29832464, 0.70167536],
       [0.36085382, 0.6391462 ],
       [0.16119699, 0.83880305],
       [0.44198388, 0.5580162 ],
       [0.3149867 , 0.6850133 ],
       [0.3830521 , 0.61694795],
       [0.5492716 , 0.45072842],
       [0.52733505, 0.47266492],
       [0.28339523, 0.71660477]], dtype=float32), array([[0.7612086 , 0.23879144],
       [0.32955885, 0.67044115],
       [0.19030444, 0.80969554],
       [0.30184534, 0.6981547 ],
       [0.61899096, 0.3810091 ],
       [0.28577438, 0.7142256 ],
       [0.28993016, 0.71006984],
       [0.19481227, 0.80518776],
       [0.39345038, 0.6065496 ],
       [0.5421205 , 0.45787948]], dtype=float32), array([[0.41676193, 0.5832381 ],
       [0.44689116, 0.5531088 ],
       [0.55712295, 0.44287702],
       [0.7086743 , 0.29132572],
       [0.22817498, 0.771825  ],
       [0.3407349 , 0.65926516],
       [0.4152216 , 0.5847784 ],
       [0.56612086, 0.43387914],
       [0.52303684, 0.4769632 ],
       [

'\nX_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.33, random_state=42)\n\nclf = TabPFNClassifier()\n\n\nmulti_target_pfn = MultiOutputClassifier(clf, n_jobs=2)\ny_pred = multi_target_pfn.fit(X_train, y_train).predict_proba(X_test)\n\n\n\n#BR = BinaryRelevanceTabPFN()\n\n\n#results = BR.predict(X, Y)\n\n\n\n\ny_pred_df = pd.DataFrame(y_pred, columns=drugs)\n\ny_test_df = pd.DataFrame(y_test, columns=drugs)\n\n\nutils.save_multilabel(y_pred_df, y_test_df, label= "PI_test", path="./")\n\n'

In [19]:
for train, test in kf.split(X, Y):
    print("%s %s" % (train, test))

print(Y)
print(np.array(y_pred).shape)

[0 2 3 4 5 6 7 9] [1 8]
[1 2 3 4 6 7 8 9] [0 5]
[0 1 3 4 5 6 8 9] [2 7]
[0 1 2 3 5 6 7 8] [4 9]
[0 1 2 4 5 7 8 9] [3 6]
[[1 1 0]
 [1 1 1]
 [1 0 1]
 [1 0 1]
 [0 1 0]
 [1 1 1]
 [0 1 1]
 [0 0 0]
 [0 0 1]
 [1 1 0]]
(3, 10, 2)


In [35]:
#y_pred = np.array(np.array(y_pred))

y_pred_array = np.array(y_pred)

y_pred_new = np.zeros((y_pred_array[0].shape[0], len(y_pred_array)))

print(y_pred_new)

for j, clas in enumerate(y_pred_array):
    for i in range(clas.shape[0]):
        if clas[i, 1] >= 0.5:
            y_pred_new[i, j] = 1
        else:
            y_pred_new[i, j] = 0

print(y_pred_new)

[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]
[[1. 0. 1.]
 [1. 1. 1.]
 [1. 1. 0.]
 [1. 1. 0.]
 [1. 0. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [0. 1. 0.]
 [0. 1. 0.]
 [1. 0. 1.]]


In [37]:
print(len(y_pred_array))

3


In [31]:
test2 = np.zeros((2,3))

print(test2.shape)
print(np.array(test2).shape)

(2, 3)
(2, 3)


In [5]:
drugs = ["A", "B", "C"]

df = pd.DataFrame(y_pred, columns=drugs)

kfolds = np.zeros((y_pred.shape[0], 1))

k = 0

for _, test in kf.split(X, Y):
    for i in test:
        kfolds[i] = k
    k += 1


df["kFolds"] = kfolds

print(df)

for train, test in kf.split(X, Y):
    print("%s %s" % (train, test))

   A  B  C  kFolds
0  1  1  0     1.0
1  0  1  0     0.0
2  0  0  0     2.0
3  2  0  0     4.0
4  1  0  2     3.0
5  0  1  1     1.0
6  2  2  0     4.0
7  0  2  0     2.0
8  2  0  0     0.0
9  0  0  0     3.0
[0 2 3 4 5 6 7 9] [1 8]
[1 2 3 4 6 7 8 9] [0 5]
[0 1 3 4 5 6 8 9] [2 7]
[0 1 2 3 5 6 7 8] [4 9]
[0 1 2 4 5 7 8 9] [3 6]


In [11]:
y_true = np.zeros((y_pred.shape[0], Y.shape[1]))

t = 0

for _, test in kf.split(X, Y):
    for i in test:
        #print(i)
        for j in range(Y.shape[1]):
            y_true[t, j] = Y[i, j]
        t += 1


print(y_true)

print(Y)

[[1. 2. 1.]
 [0. 0. 2.]
 [2. 2. 0.]
 [0. 0. 2.]
 [2. 1. 0.]
 [1. 1. 1.]
 [0. 2. 1.]
 [2. 0. 0.]
 [0. 0. 2.]
 [1. 1. 0.]]
[[2 2 0]
 [1 2 1]
 [2 1 0]
 [0 0 2]
 [0 2 1]
 [0 0 2]
 [1 1 0]
 [1 1 1]
 [0 0 2]
 [2 0 0]]
